# 0.0. Setup

In [1]:
import os
from pathlib import Path

import re
import shutil
import pickle
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

from difflib import SequenceMatcher
from sklearn.impute import SimpleImputer, KNNImputer

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / 'data'
DATA_PUBLIC_DIR = PROJECT_DIR / 'data' / 'public'
DATA_PRIVATE_DIR = PROJECT_DIR / 'data' / 'private'
DATA_PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

OUTPUT_DIR = PROJECT_DIR / 'output'
if not os.path.exists(OUTPUT_DIR):
    os.mkdir(OUTPUT_DIR)

# 1.0. Load data

In [2]:
!ls '{DATA_PUBLIC_DIR}'

'Copy of SY 2023-2024 SCHOOL LEVEL DATA ON ENROLLMENT.csv'
'SY 2022-2023 Gr 7 Enrollees.xlsx'
'SY 2022-2023 LIST OF PUBLIC SCHOOLS WITH SHIFTING SCHEDULE.xlsx'
'SY 2023-2024 Gr 7 Enrollees.xlsx'
'SY 2023-2024 LIST OF SCHOOLS WITH LONGITUDE AND LATITUDE - Copy.xlsx'
'SY 2023-2024 LIST OF SCHOOLS WITH LONGITUDE AND LATITUDE.xlsx'
'SY 2023-2024 SEAT-LEARNER RATIO.xlsx'
'SY 2024-2025 Gr 7 Enrollees.xlsx'
'SY 2024-2025 JHS ESC Tagged Learners in LIS (from their previous G6 Schools).xlsx'
'SY 2024-2025 SHS QVR Tagged Learners in LIS (from their previous G10 Schools).xlsx'
'SY 2024-2025 School Level Database WITH PSGC.xlsx'
 facilities_2023-24.csv


In [3]:
pattern = r'Gr 7'
dirpath = str(DATA_PUBLIC_DIR)
filenames = os.listdir(dirpath)
fnames_enr = [f for f in filenames if re.search(pattern, f)]
# print(fnames_enr)

fpaths_enr = [os.path.join(dirpath, f) for f in fnames_enr]
display(fpaths_enr)

school_years = [re.findall(r'SY (.*?) Gr', f)[0] for f in fnames_enr]
print(school_years)

['/workspace/project_paaral/data/public/SY 2022-2023 Gr 7 Enrollees.xlsx',
 '/workspace/project_paaral/data/public/SY 2023-2024 Gr 7 Enrollees.xlsx',
 '/workspace/project_paaral/data/public/SY 2024-2025 Gr 7 Enrollees.xlsx']

['2022-2023', '2023-2024', '2024-2025']


# 2.0. Process

## 2.1. Process one

### Function

In [4]:
def load_school_year(
    school_year: str,
    data_dir: Optional[str] = None
) -> pd.DataFrame:
    """
    Load Grade 7 enrollees data for a specific school year.

    Args:
        school_year (str): School year (e.g., 'SY 2023-2024')
        data_dir (str): Optional override for data directory

    Returns:
        DataFrame: Processed Grade 7 enrollees data
    """
    # logger.info(f"Loading Grade 7 enrollees data for {school_year}...")

    # Construct file path
    data_path = Path(data_dir) if data_dir else Path(DATA_PUBLIC_DIR)
    file_path = data_path / f"SY {school_year} Gr 7 Enrollees.xlsx"

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    # Iterate over all sheets in Excel SU
    xl = pd.ExcelFile(file_path, engine='calamine')
    # In case an Excel file has multiple sheets
    sheets = xl.sheet_names

    dfs = []
    for sheet in sheets:
        # Load Excel with selected columns only (memory efficient)
        # logger.info(f"  Reading {file_path.name}...")
        df = pd.read_excel(
            file_path,
            sheet_name=sheet,
            usecols=[
                'Region',
                'Division',
                'LRN',
                'BEIS School ID in Grade 7',
                'School Namein Grade 7',  # Note: Typo in source data - no space between "Name" and "in"
                'SY enrolled in Grade 6',
                'BEIS School ID in Grade 6',
                'School Name in Grade 6',  # Note: Correct spacing in source data
            ],
            dtype={
                'LRN': str,  # Keep as string to preserve leading zeros
                'BEIS School ID in Grade 7': str,
                'BEIS School ID in Grade 6': str,
            },
            engine='calamine'
        )
    
        # logger.info(f"  Loaded {len(df):,} student records")
    
        # Standardize column names
        df = df.rename(columns={
            'LRN': 'lrn',
            'BEIS School ID in Grade 7': 'school_id_destination',
            'School Namein Grade 7': 'school_name_destination',  # Note: Source has typo (no space)
            'BEIS School ID in Grade 6': 'school_id_origin',
            'School Name in Grade 6': 'school_name_origin',  # Note: Source has correct spacing
            'SY enrolled in Grade 6': 'sy_grade6',
            'Region': 'region',
            'Division': 'division'
        })
    
        # Add school_year column
        df['school_year'] = school_year

        dfs.append(df)
        
    dfs = pd.concat(dfs)
    
    return dfs

In [5]:
# # Uncomment to test function
# df_sy2425 = load_school_year(school_year='2024-2025')
# print(df_sy2425.shape)

In [6]:
# display(df_sy2425.head())

## 2.2. Process all

In [7]:
def _incorporate_old_region(df):
    region_6_divisions = [
        "Negros Occidental",
        "Bacolod City",
        "Bago City",
        "Cadiz City",
        "Escalante City",
        "Himamaylan City",
        "Kabankalan City",
        "La Carlota City",
        "Sagay City",
        "San Carlos City",
        "Silay City",
        "Sipalay City",
        "Talisay City",
        "Victorias City"
    ]
    
    region_7_divisions = [
        "Negros Oriental",
        "Bais City",
        "Bayawan City",
        "Canlaon City",
        "Dumaguete City",
        "Guihulngan City",
        "Tanjay City",
        "Siquijor"
    ]
        
    df['old_region'] = df['region']
    
    mask_r6 = df['division'].isin(region_6_divisions)
    df.loc[mask_r6, 'old_region'] = 'Region VI'
    
    mask_r7 = df['division'].isin(region_7_divisions)
    df.loc[mask_r7, 'old_region'] = 'Region VII'

    print(f"Created old_region column for NIR divisions previously in R6 & R7.")
    return df

In [8]:
def load_all_years(
    years: Optional[List[str]] = None,
    data_dir: Optional[str] = None
) -> pd.DataFrame:
    """
    Load and concatenate all Grade 7 enrollees data across school years.

    Args:
        years (list): List of school years to load (default: all available)
        data_dir (str): Optional override for data directory

    Returns:
        DataFrame: Combined Grade 7 enrollees data across all years
    """
    # logger.info("Loading all Grade 7 enrollees data...")

    # Default years
    if years is None:
        years = ['2022-2023', '2023-2024', '2024-2025']

    # Load each year
    dfs = []
    for year in years:
        try:
            df_year = load_school_year(year, data_dir)
            dfs.append(df_year)
            print(f"  Loaded {year}: {len(df_year):,} records")
        except FileNotFoundError:
            print(f"  File not found for {year}, skipping")
            # pass

    # Concatenate
    if not dfs:
        raise ValueError("No data files found")

    combined = pd.concat(dfs, ignore_index=True)
    # logger.info(f"Combined data: {len(combined):,} total records across {len(dfs)} years")
    combined = _incorporate_old_region(combined)

    # self.gr7_data = combined
    return combined

In [9]:
%%time
dfs_g7 = load_all_years(data_dir=DATA_PUBLIC_DIR)
print(dfs_g7.shape)

  Loaded 2022-2023: 2,136,862 records
  Loaded 2023-2024: 1,786,442 records
  Loaded 2024-2025: 2,117,984 records
Created old_region column for NIR divisions previously in R6 & R7.
(6041288, 10)
CPU times: user 1min 5s, sys: 8.05 s, total: 1min 13s
Wall time: 1min 18s


In [10]:
display(dfs_g7.head(3))

,region,division,lrn,school_id_destination,school_name_destination,sy_grade6,school_id_origin,school_name_origin,school_year,old_region
0,Region I,Ilocos Norte,100160150023,300001,Adams National High School,2021,100160,Adams CES,2022-2023,Region I
1,Region I,Ilocos Norte,100160150024,300001,Adams National High School,2021,100160,Adams CES,2022-2023,Region I
2,Region I,Ilocos Norte,100160150025,300001,Adams National High School,2021,100160,Adams CES,2022-2023,Region I


In [11]:
dfs_g7['region'].unique()

array(['Region I', 'Region II', 'Region III', 'Region IV-A', 'MIMAROPA',
       'Region V', 'Region VI', 'Region VII', 'Region VIII', 'Region IX',
       'Region X', 'Region XI', 'Region XII', 'CARAGA', 'BARMM', 'CAR',
       'NCR', 'PSO', 'NIR'], dtype=object)

In [12]:
if 'old_region' in dfs_g7.columns:
    display(dfs_g7['old_region'].unique())
    display(dfs_g7[dfs_g7['region'] == 'NIR'].sample(5))

array(['Region I', 'Region VI', 'Region II', 'Region III', 'Region IV-A',
       'MIMAROPA', 'Region V', 'Region VII', 'Region VIII', 'Region IX',
       'Region X', 'Region XI', 'Region XII', 'CARAGA', 'BARMM', 'CAR',
       'NCR', 'PSO'], dtype=object)

,region,division,lrn,school_id_destination,school_name_destination,sy_grade6,school_id_origin,school_name_origin,school_year,old_region
5995204,NIR,Negros Occidental,117323170044,404087,Our Lady of Lourdes Academy (Pul),2023,117323,Pulupandan East ES,2024-2025,Region VI
6036513,NIR,Victorias City,440510150070,302695,Victorias National High School,2023,117447,Victorias North ES,2024-2025,Region VI
6028471,NIR,Kabankalan City,117832160006,306357,Andres Trio Monteclaro National High School,2023,117832,Basak ES,2024-2025,Region VI
6039039,NIR,Canlaon City,120198150041,312964,Bayog National High School,2023,120198,Bayog ES,2024-2025,Region VII
5958800,NIR,Siquijor,120707170002,303300,Tambisan National High School,2023,120707,Solangon Elementary School,2024-2025,Region VII


In [13]:
dfs_g7.dtypes

region                     object
division                   object
lrn                        object
school_id_destination      object
school_name_destination    object
sy_grade6                   int64
school_id_origin           object
school_name_origin         object
school_year                object
old_region                 object
dtype: object

### Save output

In [15]:
for col in dfs_g7.columns:
    dfs_g7[col] = dfs_g7[col].astype(str)

outpath = os.path.join(OUTPUT_DIR, 'processed_grade_7_enrollment.parquet')

if os.path.exists(outpath):
    if os.path.isdir(outpath):
        shutil.rmtree(outpath)
        print(f"Removed directory with path {outpath}")
    else:
        os.remove(outpath)
        print(f"Removed file with path {outpath}")

dfs_g7.to_parquet(outpath, partition_cols='school_year')

print(f"Saved to parquet at {outpath}")

Removed directory with path /workspace/project_paaral/output/processed_grade_7_enrollment.parquet
Saved to parquet at /workspace/project_paaral/output/processed_grade_7_enrollment.parquet


### Load

In [16]:
outpath = os.path.join(OUTPUT_DIR, 'processed_grade_7_enrollment.parquet')
df_g7 = pd.read_parquet(outpath)
print(df_g7.shape)

(6041288, 10)


In [17]:
display(df_g7.head())

,region,division,lrn,school_id_destination,school_name_destination,sy_grade6,school_id_origin,school_name_origin,old_region,school_year
0,Region I,Ilocos Norte,100160150023,300001,Adams National High School,2021,100160,Adams CES,Region I,2022-2023
1,Region I,Ilocos Norte,100160150024,300001,Adams National High School,2021,100160,Adams CES,Region I,2022-2023
2,Region I,Ilocos Norte,100160150025,300001,Adams National High School,2021,100160,Adams CES,Region I,2022-2023
3,Region I,Ilocos Norte,100160150021,300001,Adams National High School,2021,100160,Adams CES,Region I,2022-2023
4,Region I,Ilocos Norte,100160150019,300001,Adams National High School,2021,100160,Adams CES,Region I,2022-2023


# 3.0. Crosscheck with enrollment

## 3.1. Load enrollment

In [18]:
# School enrollment
fname = 'processed_project_bukas_school_enrollment.parquet'
fpath = os.path.join(OUTPUT_DIR, fname)
sch_enr = pd.read_parquet(fpath)
print(sch_enr.shape)

(1506312, 6)


In [19]:
# Pivot enrollment
mask_sy = sch_enr['school_year'] == '2023_24'
sy_enr = sch_enr.loc[mask_sy].copy()
print(sy_enr.shape)

(502210, 6)


In [22]:
pvt_enr = sy_enr.pivot_table(
    index='school_id',
    columns='grade_level',
    values='enrollment_count',
    aggfunc='sum',
    observed=False
)
print(pvt_enr.shape)

rel_cols = ['grade_6','grade_7']
pvt_enr = pvt_enr[rel_cols]

mask_g7 = pvt_enr['grade_7'].isna()
pvt_enr = pvt_enr.loc[~mask_g7]

pvt_enr.columns = ['enr_'+col for col in pvt_enr.columns]

print(pvt_enr.shape)

(60167, 15)
(16376, 2)


In [23]:
display(pvt_enr)
# 46 makes this correct, FYI

,enr_grade_6,enr_grade_7
school_id,,
300000,NaN,38.0
300001,NaN,46.0
300002,NaN,352.0
300003,NaN,159.0
300004,NaN,64.0
...,...,...
700031,9.0,14.0
700032,98.0,110.0
700033,119.0,108.0


## 3.2. Join G7

In [58]:
g7_agg = df_g7.groupby(
    ['school_id_destination','school_year'],
    as_index=False,
    observed=False
)['lrn'].size()

g7_agg.rename(
    columns={'school_id_destination':'school_id', 'size':'enrollment'},
    inplace=True
)
g7_agg.set_index('school_id', inplace=True)

mask_sy = (g7_agg['school_year'] == '2023-2024') & (g7_agg['enrollment'] != 0)
g7_agg = g7_agg.loc[mask_sy]

print(g7_agg.shape)

(16037, 2)


In [59]:
display(g7_agg.head(3))

,school_year,enrollment
school_id,,
300000,2023-2024,36
300001,2023-2024,42
300002,2023-2024,349


In [69]:
pvt_g7 = g7_agg.pivot_table(
    index='school_id',
    columns='school_year',
    values='enrollment',
    aggfunc='sum',
    observed=False
)
pvt_g7.drop(columns=['2022-2023','2024-2025'], inplace=True)
pvt_g7.columns = ['enr_g7_'+col for col in pvt_g7.columns]

jn_g7 = pvt_enr.join(pvt_g7, how='left')
print(jn_g7.shape)

(16376, 3)


## 3.3. Comparison

In [71]:
# The values between "enr_grade_7" and "enr_g7_2023-2024" should be close or, at best, equal
display(jn_g7)

,enr_grade_6,enr_grade_7,enr_g7_2023-2024
school_id,,,
300000,NaN,38.0,36.0
300001,NaN,46.0,42.0
300002,NaN,352.0,349.0
300003,NaN,159.0,154.0
300004,NaN,64.0,63.0
...,...,...,...
700031,9.0,14.0,14.0
700032,98.0,110.0,105.0
700033,119.0,108.0,100.0
